# GCMS Spectrum Extractor — Demo

This notebook runs the deterministic extraction pipeline on a single sample
GC-MS PDF and shows the extracted peak data plus the visual overlay inline.
No model, no training, no GPU — the extraction is fully deterministic.

Pipeline: **vector** extraction (PDF drawing layer) → **raster** fallback
(OpenCV) → **text** fallback (NIST peak list).

In [ ]:
import sys, json
from pathlib import Path

sys.path.insert(0, "src")
from imgprocess import process_pdf

# Any PDF in the bundled demo folder works
pdf = Path("data/samples/_demo/100A.pdf")
print("Input PDF:", pdf)

## 1. Run the extractor

`process_pdf` returns a record with the compound name, the detected peaks,
which pipeline succeeded (`method`), any validation `warnings`, and timing.

In [ ]:
result = process_pdf(pdf, write_visual=True)

summary = {k: v for k, v in result.items() if k != "peaks"}
print(json.dumps(summary, indent=2))

In [ ]:
print(f"{result['bar_count']} peaks detected. First 12:\n")
print(f"  {'m/z':>6}   {'intensity':>9}")
print(f"  {'-'*6}   {'-'*9}")
for p in result["peaks"][:12]:
    print(f"  {p['mz']:>6}   {p['intensity']:>9}")

## 2. Visual overlay

The overlay confirms the extraction visually:

- **Red horizontal line** = detected x-axis (baseline)
- **Blue vertical line** = detected y-axis
- **Green dot** = axis origin
- **Orange marks** = detected x-axis tick labels (calibration)
- **Red dots** = detected bar tops (the extracted peaks)

Every red dot should sit on top of a real bar in the chart.

In [ ]:
from IPython.display import Image
Image(filename=f"visuals/{pdf.stem}.png")

## 3. What happens at scale

Run on the full corpus, this pipeline extracted ~48,580 spectra and **48,574
(100.0%) matched their NIST reference spectrum at cosine similarity ≥ 0.99** —
large-scale empirical proof that the extraction is correct. See the
[README](README.md) Validation section, and reproduce with:

```bash
python src/imgprocess.py --input data/samples --per-file-json
python src/sanitycheck.py        # per-file physics/consistency checks
python src/compare_peaks.py      # cosine similarity vs NIST (needs NIST library)
```